[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C39_Distributed_Training_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy 在**单进程内模拟多个 rank**，再与**单卡参考**实现 **对拍**。

这个 notebook 做三件事：① 确认环境；② 建立全课统一的「**把 W 个 rank 模拟成一个列表/多一维数组**」的心智模型与小工具；③ 立下全课的纪律——**对拍（differential testing）**。

> 心智模型：**一个 rank = 列表里的一个元素；all-reduce = 把列表求和；分片 = 切数组；通信量 = 数出来的字节**。我们写的是并行*结构与账*，不是性能。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画扩展曲线）。无需多 GPU / NCCL / torch。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅  —— 本课不需要 GPU / NCCL / torch.distributed')

## 2 · 为什么需要分布式：一个模型放不下单卡

先用一个**真实**的显存账看清动机。混合精度 + Adam 下，每个参数训练时约占 **16 字节**（fp16 参数 2 + fp16 梯度 2 + fp32 master 4 + Adam 一阶/二阶动量 4+4）。

算一个 7B / 70B 模型**仅状态**（不含激活）要多少显存，对比单卡 80GB。

In [ ]:
BYTES_PER_PARAM = 16          # fp16 参数2 + fp16 梯度2 + fp32 master4 + Adam m4 + v4
GPU_MEM_GB = 80               # 一张 A100/H100 80GB
for name, P in [('GPT-2 (1.5B)', 1.5e9), ('Llama-2-7B', 6.7e9), ('Llama-2-70B', 6.9e10)]:
    gb = P * BYTES_PER_PARAM / 1e9
    n_gpu = int(np.ceil(gb / GPU_MEM_GB))
    print(f'{name:14s} 仅训练状态 {gb:8.1f} GB  ->  至少需要 {n_gpu:3d} 张 80GB 卡只为放下状态')
# 70B 的训练状态就超过单卡，必须分片/并行
assert 6.9e10 * BYTES_PER_PARAM / 1e9 > GPU_MEM_GB, '70B 的状态应超过单卡 80GB'
print('\n结论：大模型连*状态*都放不下单卡（还没算激活）-> 必须把模型/状态切到多卡。')

## 3 · 全课统一原语：把 W 个 rank 模拟成一个列表

我们用 **长度 W 的列表** `shards`，第 `r` 个元素 = 第 `r` 个 rank 持有的张量。

在这个表示下，集合通信就是对列表的纯函数操作。先实现两个最基本的：**all-reduce(sum)** 与 **broadcast**。

In [ ]:
def all_reduce_sum(shards):
    '''每个 rank 贡献一个同形张量，求和后每个 rank 都拿到同一个和。
       真实世界：dist.all_reduce(t, op=SUM)。这里：把列表逐元素加起来再广播。'''
    total = np.zeros_like(shards[0])
    for t in shards:                 # 模拟每个 rank 把自己的贡献加进来
        total = total + t
    return [total.copy() for _ in shards]   # 每个 rank 都得到同一个结果

def broadcast(value, world_size, src=0):
    '''把 src rank 的张量复制给所有 rank。'''
    return [np.array(value).copy() for _ in range(world_size)]

W = 4
rng = np.random.default_rng(0)
shards = [rng.standard_normal(5) for _ in range(W)]
out = all_reduce_sum(shards)
ref = sum(shards)                    # 单进程参考：直接求和
assert all(np.allclose(o, ref) for o in out), 'all-reduce 后每个 rank 应持有同一个和'
print('all-reduce(sum) 4 个 rank -> 每个 rank 都拿到全局和 ✅')
print('每个 rank 的结果一致:', np.allclose(out[0], out[1]) and np.allclose(out[1], out[2]))

## 4 · 立纪律：对拍（differential testing）—— 数据并行的第一个例子

数据并行的**正确性契约**：每个 rank 用自己那份数据算梯度，**all-reduce 求平均**后，应当**等于把所有数据拼成一个大 batch 在单卡上算的梯度**。

用最简单的线性回归 `loss = ||Xw - y||² / N` 验证它（梯度有闭式：`g = (2/N) Xᵀ(Xw - y)`）。

In [ ]:
def linreg_grad(X, y, w):
    '''MSE 损失对 w 的梯度，按样本数归一化。'''
    N = X.shape[0]
    resid = X @ w - y
    return (2.0 / N) * (X.T @ resid)

rng = np.random.default_rng(1)
W_ranks = 4
d = 3
local_n = 8                          # 每个 rank 的本地 batch
w = rng.standard_normal(d)
# 给每个 rank 切一份不同数据
Xs = [rng.standard_normal((local_n, d)) for _ in range(W_ranks)]
ys = [rng.standard_normal(local_n) for _ in range(W_ranks)]

# 分布式：每个 rank 算本地梯度，再 all-reduce 求平均
local_grads = [linreg_grad(Xs[r], ys[r], w) for r in range(W_ranks)]
summed = all_reduce_sum(local_grads)[0]
dp_grad = summed / W_ranks           # 平均 = (1/W) Σ g_r

# 单卡参考：把所有数据拼成一个大 batch
X_big = np.concatenate(Xs, axis=0)
y_big = np.concatenate(ys, axis=0)
single_grad = linreg_grad(X_big, y_big, w)

err = np.max(np.abs(dp_grad - single_grad))
print(f'DP 平均梯度 vs 单卡大 batch 梯度: max|err| = {err:.2e}')
assert np.allclose(dp_grad, single_grad, atol=1e-12), 'DP 平均梯度应等于单卡大 batch 梯度'
print('✅ 数据并行的正确性契约成立：平均梯度 == 单卡大 batch 梯度')
print('（成立的前提：每个 rank 的 local_n 相同，所以等权平均恰好等于全局平均）')

## 5 · 通信也有账：数出 all-reduce 搬了多少字节

分布式的另一半是**成本**。我们后面会反复用「数出来的字节数」当通信量。

先建立直觉：**朴素 all-reduce**（汇总到 rank0 再广播回去）的通信量随 W 增长；而 **ring all-reduce**（模块 01 细讲）每个 rank 收发约 `2(W-1)/W × 张量大小`，与 W 几乎无关。

In [ ]:
def naive_allreduce_bytes(tensor_bytes, W):
    '''朴素：W-1 个 rank 各把数据发给 rank0(reduce)，rank0 再广播给 W-1 个 rank。'''
    reduce_phase = (W - 1) * tensor_bytes      # 都发给 root
    broadcast_phase = (W - 1) * tensor_bytes   # root 发回去
    return reduce_phase + broadcast_phase      # 总搬运量 = 2(W-1)*S

def ring_allreduce_bytes_per_rank(tensor_bytes, W):
    '''ring：每个 rank 在 reduce-scatter 与 all-gather 各 W-1 步、每步发 S/W。'''
    return 2 * (W - 1) * (tensor_bytes / W)     # ≈ 2S，与 W 几乎无关

S = 100e6                                       # 100 MB 的梯度张量
print(f"{'W':>4} {'朴素总量(MB)':>14} {'ring每rank(MB)':>16}")
for W_ in [2, 4, 8, 16, 64]:
    nv = naive_allreduce_bytes(S, W_) / 1e6
    rg = ring_allreduce_bytes_per_rank(S, W_) / 1e6
    print(f'{W_:>4} {nv:>14.0f} {rg:>16.0f}')
# ring 每 rank 的收发量上界是 2S，与 W 无关（朴素的总量却线性涨）
assert ring_allreduce_bytes_per_rank(S, 1024) < 2 * S
print('\n✅ 关键：ring all-reduce 每 rank 通信量≈2S 不随 W 爆炸 —— 这是大规模数据并行可行的根基（模块 01 细讲）。')

## 6 · 一个会贯穿全课的对拍工具

把「对拍」封装成小函数，后面每个模块都用它判定「我的分布式实现 == 单卡参考」。它是本课所有 `assert` 背后的统一裁判。

In [ ]:
def check_allclose(name, got, ref, atol=1e-10):
    '''对拍：分布式实现结果 vs 单卡/单进程参考。返回是否一致并打印。'''
    got = np.asarray(got); ref = np.asarray(ref)
    ok = np.allclose(got, ref, atol=atol)
    max_err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参考不一致！'
    return ok

# 演示：reduce-scatter + all-gather 应当 == all-reduce（模块 01 的核心恒等式）
def reduce_scatter_then_all_gather(shards):
    W = len(shards)
    n = shards[0].shape[0]
    assert n % W == 0, '演示用，长度需被 W 整除'
    total = sum(shards)                          # 先全局求和
    chunks = np.split(total, W)                  # reduce-scatter：每 rank 拿一片
    gathered = np.concatenate(chunks)           # all-gather：拼回完整
    return [gathered.copy() for _ in range(W)]

shards = [rng.standard_normal(8) for _ in range(4)]
check_allclose('rs+ag == all-reduce', reduce_scatter_then_all_gather(shards)[0], sum(shards))
print('\n这就是全课的工作流：写分布式实现 -> 对拍单卡/单进程参考 -> assert 兜底。')

## 7 · 旁注：真实 torch.distributed 长什么样

本课模拟的 all-reduce，在真实多 GPU 上是这样（伪代码，**本环境不跑**）：

```python
import torch, torch.distributed as dist

dist.init_process_group(backend='nccl')        # 每个进程一张 GPU
rank = dist.get_rank(); world = dist.get_world_size()
torch.cuda.set_device(rank % torch.cuda.device_count())

grad = compute_local_gradient(...)             # 本 rank 的本地梯度
dist.all_reduce(grad, op=dist.ReduceOp.SUM)    # == 我们的 all_reduce_sum
grad /= world                                  # 求平均 == 我们的 /W
# 启动：torchrun --nproc_per_node=8 train.py
```

对应关系：`dist.all_reduce(SUM)` ↔ 我们的 `all_reduce_sum`、`/world` ↔ 我们的 `/W`、`rank/world_size` ↔ 列表下标与长度。框架把「跨网络、并发、用 ring 算法」自动处理掉——你只需写对**聚合逻辑与账**，正是本课练的东西。

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 里写出的每个 all-reduce / 分片 / 并行切分 / 调度 / 恢复逻辑，都会用 `np.allclose` 对拍单卡参考；结构正确则数值一致，数值一致则逻辑可迁移到 `torch.distributed` / FSDP / Megatron。

**接下来六个模块**：01 集合通信 → 02 数据并行/FSDP → 03 张量/流水并行 → 04 checkpoint/容错 → 05 编排/调试。每一步都建立在前一步之上。

下一站：**模块 01 · 集合通信原语** —— 所有并行共用的通信积木。